# 05 — Jargon glossary

Every term used casually in this repo's docs, logs, and findings — grouped by
theme, with real numbers from our own recorded data where that helps.

## A. Market-data terms

| term | meaning |
|---|---|
| **tick / trade print** | one executed trade: price, size, timestamp |
| **quote** | the current best **bid** (highest buyer) and **ask** (lowest seller), with sizes |
| **mid** | `(bid + ask) / 2` — fair-value estimate you cannot actually trade at |
| **spread** | `ask − bid` — the market maker's toll; you pay ~half on entry, ~half on exit |
| **bps — basis points** | 1 bp = 0.01% = 1/10,000. THE unit of this project: spreads, returns, costs, thresholds are all quoted in bps because at 5–60s horizons everything is small |
| **bar / candle** | open/high/low/close/volume aggregated over an interval (1 min here) |
| **aggressor / taker side** | which side *initiated* a trade (buyer lifted the ask vs seller hit the bid). Crypto feeds tag it; equities feeds don't, so we infer it from position vs mid (the *tick rule* / Lee-Ready) |
| **microprice** | size-weighted quote: `(bid_size·ask + ask_size·bid) / (bid_size+ask_size)` — where the book is "leaning"; a classic next-move predictor |
| **book imbalance** | `(bid_size − ask_size) / total` — buy pressure vs sell pressure at the top of the book |
| **IEX vs SIP** | US equities tape sources. SIP = consolidated all-exchange feed (paid); IEX = one exchange's slice (free — what Alpaca's free tier streams). Fine for research, thinner than full reality |

Some real numbers — why bps is the right unit:

In [1]:
from pathlib import Path
import duckdb

def spread_stats(db, label):
    conn = duckdb.connect(str(Path("..") / "data" / db), read_only=True)
    df = conn.execute('''
        SELECT symbol,
               round(median((ask - bid) / ((ask+bid)/2)) * 1e4, 2) AS median_spread_bps,
               round(avg((ask - bid)) , 4) AS avg_spread_abs
        FROM quotes GROUP BY symbol ORDER BY 2''').df()
    conn.close()
    df["session"] = label
    return df

import pandas as pd
pd.concat([
    spread_stats("equities_2026-07-06.duckdb", "equities (IEX)"),
    spread_stats("session.duckdb", "crypto"),
])[["session", "symbol", "median_spread_bps", "avg_spread_abs"]]

,session,symbol,median_spread_bps,avg_spread_abs
0,equities (IEX),SPY,0.40,0.0331
1,equities (IEX),NVDA,1.02,0.0736
2,equities (IEX),AAPL,1.61,0.1503
0,crypto,ETH/USD,11.19,1.9631
1,crypto,BTC/USD,11.42,70.6125


Read that table as **the toll booth**: a round trip costs roughly the full
spread (+ fees). SPY at ~1 bp is ~20× cheaper to trade than BTC at ~20 bps —
which is why a mediocre model loses hundreds of bps on crypto and single
digits on equities with the *same* predictions.

## B. Prediction & evaluation terms

| term | meaning |
|---|---|
| **horizon (k)** | how far ahead we predict: "10s horizon" = predict the mid's return from now to now+10s |
| **forward return / label** | what actually happened over the horizon — only knowable *after* it elapses |
| **lookahead (bias)** | accidentally letting the model see the future (e.g. training at time t on a label that needs t+10s data). The #1 way backtests lie. Our `LabelQueue` makes it structurally impossible |
| **walk-forward** | evaluating strictly in time order — the model at 3pm has only seen data before 3pm. Never shuffle time series |
| **train/serve skew** | live code and backtest code differing subtly so backtest results don't transfer. Our guard: replay and live share `SymbolPipeline` verbatim |
| **overlapping windows / the overlap trap** | at 6 quotes/s with a 10s horizon, consecutive predictions share ~99% of their outcome window — they're not independent samples, and accuracy on them mostly measures autocorrelation. Honest scoring spaces predictions ≥ one horizon apart ("non-overlapping" / independent windows) |
| **directional accuracy (dir)** | fraction of predictions with the correct *sign*. 0.5 = coin flip |
| **baselines: zero / persistence / fade** | rules any model must beat: predict 0 (MAE floor); "last move continues" (persistence); "last move reverses" (fade). `d-best` in our tables = model minus the best of these |
| **MAE edge** | how much smaller the model's mean absolute error is than always-predicting-zero. Positive = the model's *magnitudes* carry information |
| **mean reversion vs momentum** | does a move tend to reverse (fade wins) or continue (persistence wins)? Our finding: everything we measured mean-reverts at 5–10s, strongest in the least liquid names |
| **regime** | a period where the market behaves one way (quiet, trending, volatile). Online learning's selling point is adapting when regimes shift |
| **ablation** | knock out a feature group, re-run, see what breaks — how we learned momentum features carry the signal on crypto |

In [2]:
# Mean reversion, shown raw: sample BTC mids every 10s and ask
# "did the next 10s move have the same sign as the last one?"
from pathlib import Path
import duckdb, numpy as np

conn = duckdb.connect(str(Path("..") / "data" / "paper_2026-07-05.duckdb"), read_only=True)
q = conn.execute("SELECT ts_ns, (bid+ask)/2 AS mid FROM quotes WHERE symbol='BTC/USD' ORDER BY ts_ns").df()
conn.close()

# resample to one mid per 10s bucket
q["bucket"] = (q.ts_ns // 10_000_000_000).astype("int64")
mids = q.groupby("bucket").mid.last()
rets = mids.pct_change().dropna()
rets = rets[rets != 0]
same_sign = (np.sign(rets) == np.sign(rets.shift(1))).dropna()
print(f"{len(rets)} consecutive 10s BTC returns")
print(f"persistence (same sign as last): {same_sign.mean():.3f}")
print(f"fade        (opposite sign)    : {1 - same_sign.mean():.3f}   <- mean reversion")

12590 consecutive 10s BTC returns
persistence (same sign as last): 0.439
fade        (opposite sign)    : 0.561   <- mean reversion


## C. Trading & execution terms

| term | meaning |
|---|---|
| **paper trading** | simulated execution against real market prices with fake money — Alpaca's paper API mirrors its real one, so the same code path goes live later |
| **round trip** | enter + exit one position. Costs ≈ full spread + 2× fees + slippage |
| **slippage** | the difference between the price you saw and the price you got (market moved, or your order ate into the book) |
| **notional** | position size in dollars (qty × price) |
| **fixed-fractional sizing** | risk a fixed % of equity per trade (we use 1%, capped) |
| **vol-scaled sizing** | shrink positions when volatility is high so each trade risks similar dollars |
| **dead zone** | a "too small to care" band around zero — predictions inside it are treated as no-signal (fights overtrading) |
| **threshold / cost gate** | trade only if predicted move > fee + half-spread + dead-zone. Why our honest configs traded zero times on crypto |
| **overtrading** | trading so often that costs eat you even when direction is decent — see NVDA: 489 trades × ~1.5 bps spread = dead |
| **hit rate** | fraction of round trips with positive net PnL |
| **PnL (realized / unrealized)** | profit & loss: realized = closed positions; unrealized = open position marked to market |
| **circuit breaker / daily-loss limit** | hard stop: lose more than $X today → no new entries until tomorrow |
| **kill switch / flatten** | close everything, cancel all orders, immediately (`flatten_all`) |
| **long-only** | can buy and sell what you hold, but not short (Alpaca crypto accounts are non-marginable, hence ours) |
| **meta-labeling** | second model that predicts *whether the first model is right*, gating/scaling its signals — improves calibration, cuts bad trades |

## D. Infrastructure & ops terms

| term | meaning |
|---|---|
| **soak (test)** | run the whole system continuously for a long period (hours/days) watching for leaks, drift, disconnects — from hardware "soak testing". Our 46h run = a soak |
| **hot path / cold path** | code on the tick→decision route (µs budget, no pandas, no disk reads) vs everything offline (reports, notebooks, anything goes) |
| **backpressure** | bounded queues make a slow consumer automatically slow the producer, instead of memory growing until the process dies |
| **high-water mark (hwm)** | the worst backlog a queue ever reached — our early-warning gauge for a consumer falling behind |
| **ring buffer** | fixed-size circular array: constant memory, O(1) append, oldest data overwritten — how the hot path remembers recent prices |
| **WAL (write-ahead log)** | DuckDB's crash-safety journal (`.wal` file); replayed automatically on next connect if the writer died |
| **heartbeat / ping-pong** | periodic keepalive frames on a WebSocket; silence = dead connection = reconnect |
| **exponential backoff** | wait 1s, 2s, 4s, ... between reconnect attempts so a struggling server isn't hammered (plus jitter so many clients don't sync up) |
| **WS connection limit** | Alpaca free tier: ONE stream connection per feed — why the server and laptop can't both stream crypto |
| **feed → local latency** | exchange event timestamp vs when *we* received it (network + feed delay); ~35ms Mac/UK, ~100ms Hetzner/DE. Distinct from **decision latency** (our processing: ~10–200µs) |
| **systemd service / linger** | Linux's process manager keeps the pipeline running and restarts it on crash; "linger" lets a user's services run without an SSH session |
| **cron** | time-scheduled jobs (`30 14 * * 1-5` = 14:30 UTC weekdays) — runs the equities recording and nightly report |
| **train/serve parity redeploy** | pull main → run tests on the server → restart service. One line in docs/DEPLOY.md |

## E. Project shorthand you'll see in findings

- **`d-best`** — model directional accuracy minus the best naive baseline. The one-number answer to "is there any skill here?"
- **`edge%`** — MAE edge vs the zero baseline (magnitude skill).
- **`q_hwm`** — queue high-water mark (see backpressure).
- **`proc_us`** — per-event processing time in microseconds, tracked against the <15ms budget.
- **"the toll booth"** — round-trip trading costs; **"fighting efficiency"** — when costs are near zero but the market's just hard to predict (SPY).
- **"plumbing vs edge"** — the system works (plumbing) ≠ the strategy makes money (edge). We claim the first, measure the second honestly, and currently have: plumbing yes, edge no.

In [3]:
# The glossary's punchline, computed live: the toll booth vs typical moves.
from pathlib import Path
import duckdb
import numpy as np

rows = []
for db, sym in [("paper_2026-07-05.duckdb", "BTC/USD"), ("equities_2026-07-06.duckdb", "SPY")]:
    conn = duckdb.connect(str(Path("..") / "data" / db), read_only=True)
    q = conn.execute(f"SELECT ts_ns, (bid+ask)/2 AS mid, (ask-bid)/((ask+bid)/2)*1e4 AS spr "
                     f"FROM quotes WHERE symbol='{sym}' ORDER BY ts_ns").df()
    conn.close()
    q["bucket"] = (q.ts_ns // 10_000_000_000).astype("int64")
    mids = q.groupby("bucket").mid.last()
    move_bps = (mids.pct_change().abs() * 1e4).median()
    rows.append({"symbol": sym, "median 10s |move| (bps)": round(move_bps, 2),
                 "median spread (bps)": round(q.spr.median(), 2),
                 "move / toll ratio": round(move_bps / q.spr.median(), 2)})
import pandas as pd
pd.DataFrame(rows)

,symbol,median 10s |move| (bps),median spread (bps),move / toll ratio
0,BTC/USD,1.64,11.37,0.14
1,SPY,0.40,0.40,1.00


If the typical move is *smaller* than the toll, no direction-guesser can win —
you need magnitude selectivity (trade rarely, only the big ones). That single
ratio explains most of this project's findings so far.